In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2
RANDOM_SEED = 123

In [3]:
rng = np.random.default_rng(RANDOM_SEED)

In [4]:
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6

In [5]:
df = pd.read_csv("padufes20_metadata_official.csv")

In [6]:
df.columns

Index(['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father',
       'background_mother', 'age', 'pesticide', 'gender',
       'skin_cancer_history', 'cancer_history', 'has_piped_water',
       'has_sewage_system', 'fitspatrick', 'region', 'diameter_1',
       'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed',
       'elevation', 'img_id', 'biopsed'],
      dtype='str')

In [7]:
keep_cols = [
    "img_id",
    "diagnostic",
    "patient_id",
    "age",
    "gender",
    "region",
]
df = df[keep_cols]

In [8]:
df["diagnostic"].value_counts(dropna="FALSE")

diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64

In [9]:
df["dataset"] = "padufes20"

In [10]:
df.rename(columns={
    "img_id": "image_name",
    "diagnostic": "label",
    "gender": "sex",
    "region": "location",
}, inplace=True)

In [11]:
df["sex"] = df["sex"].fillna("unknown")

In [12]:
df.isna().sum()

image_name    0
label         0
patient_id    0
age           0
sex           0
location      0
dataset       0
dtype: int64

In [13]:
assert df.isnull().sum().sum() == 0

In [14]:
assert len(df) == df["image_name"].nunique()

In [15]:
missing = []
for f in df["image_name"]:
    if not os.path.exists(os.path.join("/home/prasad/datasets/padufes20/images/", f)):
        missing.append(f)

print(len(missing))

0


In [16]:
sorted(df["label"].unique().tolist())

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']

In [17]:
mapping = {
    "ACK": "AKIEC",
    "BCC": "BCC",
    "MEL": "MEL",
    "NEV": "NEV",
    "SCC": "SCC",
    "SEK": "SEK",
}

In [18]:
assert set(df["label"].unique()) == set(mapping.keys())

In [19]:
df["label"] = df["label"].map(mapping)

In [20]:
assert df["label"].isna().sum() == 0

In [21]:
assert sorted(df["label"].unique()) == [
    "AKIEC", "BCC", "MEL", "NEV", "SCC", "SEK"]

In [22]:
df["label"].value_counts()

label
BCC      845
AKIEC    730
NEV      244
SEK      235
SCC      192
MEL       52
Name: count, dtype: int64

In [23]:
print(df["label"].value_counts().sum())

2298


In [24]:
df["binary_label"]=(df["label"].isin(
    ["AKIEC", "MEL", "BCC", "SCC"])).astype(int)

In [25]:
assert set(df["binary_label"].unique()) == {0,1}

In [26]:
print(df["binary_label"].value_counts())

binary_label
1    1819
0     479
Name: count, dtype: int64


In [27]:
print(df["binary_label"].value_counts().sum())

2298


In [28]:
print(pd.crosstab(df["binary_label"], df["label"]))

label         AKIEC  BCC  MEL  NEV  SCC  SEK
binary_label                                
0                 0    0    0  244    0  235
1               730  845   52    0  192    0


In [29]:
df["sex"].value_counts()

sex
unknown    804
FEMALE     753
MALE       741
Name: count, dtype: int64

In [30]:
print(df["sex"].value_counts().sum())

2298


In [31]:
print(pd.crosstab(df["sex"], df["label"]))

label    AKIEC  BCC  MEL  NEV  SCC  SEK
sex                                    
FEMALE     153  406   28   49   90   27
MALE       130  439   24   26  102   20
unknown    447    0    0  169    0  188


In [32]:
df["age"].value_counts()

age
55    73
73    73
58    70
75    66
62    66
      ..
8      2
24     2
12     2
6      2
18     1
Name: count, Length: 84, dtype: int64

In [33]:
df["age"].value_counts().sum()

np.int64(2298)

In [34]:
df["age"].isna().sum()

np.int64(0)

In [35]:
print(pd.crosstab(df["age"], df["label"]))

label  AKIEC  BCC  MEL  NEV  SCC  SEK
age                                  
6          0    0    0    2    0    0
8          0    0    0    2    0    0
9          0    0    0    3    0    0
10         0    0    0    5    0    0
12         0    0    0    2    0    0
..       ...  ...  ...  ...  ...  ...
89         2    4    0    0    0    1
90         2    3    2    0    4    3
91         0    5    0    0    2    0
92         1    0    0    0    3    0
94         0    2    0    0    1    0

[84 rows x 6 columns]


In [36]:
df["location"].value_counts()

location
FACE       570
FOREARM    392
CHEST      280
BACK       248
ARM        192
NOSE       158
HAND       126
NECK        93
THIGH       73
EAR         73
ABDOMEN     36
LIP         23
SCALP       18
FOOT        16
Name: count, dtype: int64

In [37]:
df["location"].value_counts().sum()

np.int64(2298)

In [38]:
df["location"].isna().sum()

np.int64(0)

In [39]:
print(pd.crosstab(df["location"], df["label"]))

label     AKIEC  BCC  MEL  NEV  SCC  SEK
location                                
ABDOMEN       3    2    2   19    2    8
ARM          91   44    7   10   21   19
BACK         16   91   14   86   17   24
CHEST        60  135    5   29   20   31
EAR          11   36    2    3   13    8
FACE        139  260    5   38   29   99
FOOT          4    0    0    6    6    0
FOREARM     261   67    5   10   28   21
HAND         78   19    2    6   19    2
LIP           4   10    0    3    6    0
NECK         18   48    3    8   10    6
NOSE         23  123    0    0    9    3
SCALP         4    0    0    9    1    4
THIGH        18   10    7   17   11   10


# Split

In [40]:
len(df["patient_id"])

2298

In [41]:
patient_ids = df["patient_id"].dropna().unique().to_numpy()

In [42]:
rng.shuffle(patient_ids)

In [43]:
df["patient_id"].isna().sum()

np.int64(0)

In [44]:
n_patients = len(patient_ids)
n_train = int(TRAIN_RATIO * n_patients)
n_val = int(VAL_RATIO * n_patients)

In [45]:
train_patients = patient_ids[:n_train]
val_patients = patient_ids[n_train:n_train + n_val]
test_patients = patient_ids[n_train + n_val:]

In [46]:
train_ids = set(train_patients)
val_ids   = set(val_patients)
test_ids  = set(test_patients)

print("Train ∩ Val :", train_ids & val_ids)
print("Train ∩ Test:", train_ids & test_ids)
print("Val ∩ Test  :", val_ids & test_ids)

Train ∩ Val : set()
Train ∩ Test: set()
Val ∩ Test  : set()


In [47]:
df["split"] = "unassigned"
df.loc[df["patient_id"].isin(train_patients), "split"] = "train"
df.loc[df["patient_id"].isin(val_patients), "split"] = "val"
df.loc[df["patient_id"].isin(test_patients), "split"] = "test"

In [48]:
assert (df["split"] != "unassigned").all()

In [49]:
print("Patients per split:")
print(df.groupby("split")["patient_id"].nunique())

Patients per split:
split
test     275
train    961
val      137
Name: patient_id, dtype: int64


In [50]:
print("\nImages per split:")
print(df["split"].value_counts())


Images per split:
split
train    1620
test      457
val       221
Name: count, dtype: int64


In [51]:
print("\nClass distribution per split:")
print(pd.crosstab(df["split"], df["label"]))


Class distribution per split:
label  AKIEC  BCC  MEL  NEV  SCC  SEK
split                                
test     139  162   18   46   43   49
train    520  611   30  166  133  160
val       71   72    4   32   16   26


In [52]:
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

label  AKIEC  BCC  MEL  NEV  SCC  SEK
split                                
test     139  162   18   46   43   49
train    520  611   30  166  133  160
val       71   72    4   32   16   26


In [53]:
pd.crosstab(df["age"], df["split"])

split,test,train,val
age,,,
6,0,2,0
8,0,2,0
9,0,3,0
10,4,1,0
12,0,2,0
...,...,...,...
89,0,7,0
90,0,12,2
91,0,7,0


In [54]:
pd.crosstab(df["sex"], df["split"])

split,test,train,val
sex,,,
FEMALE,159,536,58
MALE,142,524,75
unknown,156,560,88


In [55]:
pd.crosstab(df["location"], df["split"])

split,test,train,val
location,,,
ABDOMEN,8,23,5
ARM,51,126,15
BACK,53,170,25
CHEST,41,205,34
EAR,14,55,4
FACE,126,398,46
FOOT,0,10,6
FOREARM,78,275,39
HAND,24,89,13


In [56]:
df.groupby(["split", "sex", "label"]).size()

split  sex      label
test   FEMALE   AKIEC     39
                BCC       79
                MEL       10
                NEV        8
                SCC       20
                SEK        3
       MALE     AKIEC     21
                BCC       83
                MEL        8
                NEV        1
                SCC       23
                SEK        6
       unknown  AKIEC     79
                NEV       37
                SEK       40
train  FEMALE   AKIEC    105
                BCC      295
                MEL       18
                NEV       32
                SCC       62
                SEK       24
       MALE     AKIEC     92
                BCC      316
                MEL       12
                NEV       24
                SCC       71
                SEK        9
       unknown  AKIEC    323
                NEV      110
                SEK      127
val    FEMALE   AKIEC      9
                BCC       32
                NEV        9
                SCC  

In [57]:
assert df["image_name"].is_unique
assert df["split"].isin(["train", "val", "test"]).all()
assert df["sex"].isin(["MALE", "FEMALE", "unknown"]).all()
assert df["label"].notna().all()

In [58]:
leakage = (
    df.groupby("patient_id")["split"]
    .nunique()
    .gt(1)
    .sum()
)
print(f"Patients appearing in multiple splits: {leakage}")

Patients appearing in multiple splits: 0


In [59]:
df=df[["dataset", "split", "image_name", "label", "binary_label", "patient_id",
        "age", "sex", "location"]]

In [60]:
expected_cols = [
    "dataset",
    "split",
    "image_name",
    "label",
    "binary_label",
    "patient_id",
    "age",
    "sex",
    "location",
]
assert list(df.columns) == expected_cols

In [61]:
print("=" * 60)
print("PADUFES20 metadata preprocessing completed")
print(f"Images   : {len(df)}")
print(f"Patients : {df['patient_id'].nunique()}")
print(f"Classes  : {df['label'].nunique()}")
print(df["split"].value_counts())
print("=" * 60)

PADUFES20 metadata preprocessing completed
Images   : 2298
Patients : 1373
Classes  : 6
split
train    1620
test      457
val       221
Name: count, dtype: int64


In [62]:
print(df.isnull().sum())

dataset         0
split           0
image_name      0
label           0
binary_label    0
patient_id      0
age             0
sex             0
location        0
dtype: int64


In [63]:
assert df["image_name"].is_unique
assert df["patient_id"].notna().all()
assert df["label"].notna().all()
assert df["binary_label"].isin([0, 1]).all()
assert df["split"].isin(["train", "val", "test"]).all()

In [64]:
assert df.isnull().sum().sum() == 0

In [65]:
df.to_csv("padufes20_metadata_clean.csv", index=False, encoding="utf-8")

In [66]:
df

,dataset,split,image_name,label,binary_label,patient_id,age,sex,location
0,padufes20,train,PAT_1516_1765_530.png,NEV,0,PAT_1516,8,unknown,ARM
1,padufes20,train,PAT_46_881_939.png,BCC,1,PAT_46,55,FEMALE,NECK
2,padufes20,val,PAT_1545_1867_547.png,AKIEC,1,PAT_1545,77,unknown,FACE
3,padufes20,test,PAT_1989_4061_934.png,AKIEC,1,PAT_1989,75,unknown,HAND
4,padufes20,train,PAT_684_1302_588.png,BCC,1,PAT_684,79,MALE,FOREARM
...,...,...,...,...,...,...,...,...,...
2293,padufes20,train,PAT_1708_3156_175.png,AKIEC,1,PAT_1708,73,unknown,HAND
2294,padufes20,train,PAT_46_880_140.png,BCC,1,PAT_46,55,FEMALE,CHEST
2295,padufes20,val,PAT_1343_1217_404.png,SEK,0,PAT_1343,74,unknown,FOREARM
2296,padufes20,test,PAT_326_690_823.png,BCC,1,PAT_326,58,FEMALE,FACE
